In [39]:
# Create order data and a customer table with a duplicate join key
import pandas as pd
orders = pd.DataFrame({
    "order_id": [1, 2, 3, 4],
    "customer_id": ["C01", "C02", "C01", "C03"],
    "amount": [100, 200, 150, 80],
})

customers = pd.DataFrame({
    "customer_id": ["C01", "C01", "C02", "C03"],
    "region": ["North", "North", "South", "West"],
    "account_type": ["retail", "retail", "business", "retail"],
})

In [40]:
# Capture the baseline revenue before enriching orders
original_total_revenue=orders["amount"].sum()
print("Original total order: ",original_total_revenue)

Original total order:  530


In [41]:
# Detect duplicate customer keys that could multiply order rows
duplicated_customer=customers[customers.duplicated(keep="first")]
print("Duplicated customers:\n",duplicated_customer)

Duplicated customers:
   customer_id region account_type
1         C01  North       retail


In [42]:
# Enrich orders with customer attributes using a left join
order_customer=orders.merge(
    customers,
    on="customer_id",
    how="left"
)
order_customer

,order_id,customer_id,amount,region,account_type
0,1,C01,100,North,retail
1,1,C01,100,North,retail
2,2,C02,200,South,business
3,3,C01,150,North,retail
4,3,C01,150,North,retail
5,4,C03,80,West,retail


In [43]:
# Compare revenue after the join with the original total
new_total_order=order_customer["amount"].sum()
print("New total order: ",new_total_order)

New total order:  780


### Why the Total Changed

- The `customers` table contains a duplicate `customer_id` (`C01` appears twice).
- During the merge, each order for `C01` matched both customer records.
- As a result, the orders belonging to `C01` were duplicated in the merged table.
- The duplicated rows caused the `amount` values for those orders to be counted more than once.
- Therefore, the total order amount increased after the merge.

### Conclusion

The total changed because the merge created additional rows due to duplicate customer keys in the `customers` table, resulting in inflated revenue totals.

In [49]:
# Validate key uniqueness, row counts, and revenue before trusting the join
def revenue_validation_process(customers, orders):
    customers = customers.copy()
    orders = orders.copy()
    original_total_revenue = orders["amount"].sum()

    # Validate key uniqueness before requiring a many-to-one merge.
    if not customers["customer_id"].is_unique:
        print("FAIL: Duplicate customer_id found in customers table")
        return
    print("PASS: Customer IDs are unique")

    merged = orders.merge(
        customers,
        on="customer_id",
        how="left",
        validate="many_to_one",
    )
    print("PASS: Merge validation successful")

    if len(orders) != len(merged):
        print("WARNING: Row count changed after merge")
    else:
        print("PASS: Row count unchanged")

    if original_total_revenue != merged["amount"].sum():
        print("WARNING: Revenue total changed after merge")
    else:
        print("PASS: Revenue total unchanged")


In [50]:
# Run validation against the duplicated customer table
revenue_validation_process(customers,orders)

FAIL: Duplicate customer_id found in customers table
FAIL:  Merge keys are not unique in right dataset; not a many-to-one merge


In [47]:
# Remove exact duplicate customer records before validation
valid_customers=customers.drop_duplicates().reset_index(drop=True)
valid_customers

,customer_id,region,account_type
0,C01,North,retail
1,C02,South,business
2,C03,West,retail


In [51]:
# Confirm that the cleaned customer table passes validation
revenue_validation_process(valid_customers,orders)

PASS: Customer IDs are unique
Pass: Merge validation successful
PASS: Row count unchanged
PASS: Revenue total unchanged


In [52]:
# Perform a validated many-to-one merge with one row per customer
reconciled_customers=customers.drop_duplicates(subset=["customer_id"])
reconciled_table=orders.merge(
    reconciled_customers,
    on="customer_id",
    how="left",
    validate="many_to_one"
)
reconciled_table

,order_id,customer_id,amount,region,account_type
0,1,C01,100,North,retail
1,2,C02,200,South,business
2,3,C01,150,North,retail
3,4,C03,80,West,retail


In [54]:
# Confirm that the reconciled merge preserves total revenue
final_total_revenue=reconciled_table["amount"].sum()
if final_total_revenue!=original_total_revenue:
    print("FAIL: Total revenue changed after merge")
else:
    print("PASS: Total revenue remained unchanged")

PASS: Total revenue remained unchanged


### Important notes

- A many-to-one merge requires unique `customer_id` values in the right-hand table; duplicate keys can multiply order rows and inflate revenue.
- `validate="many_to_one"` is a safeguard that raises an error instead of silently accepting an invalid relationship.
- Compare row counts and monetary totals before and after the merge; both should remain unchanged when only attributes are added.
- Remove duplicate keys only when the business rule confirms the records are duplicates; otherwise resolve the conflicting customer records first.
- Check for unmatched customer IDs after a left join because missing customer attributes can still pass row-count validation.
- Keep validation inputs local to the function rather than relying on global totals so the process is reusable.
